In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import numpy as np

import ModelFxp12 as fxp

In [ ]:
class GetChannel:
    def __init__(self, channel):
      self.channel = channel
    
    def __call__(self, img):
        return  torch.unsqueeze(img[self.channel], 0)

stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(*stats),
    #transforms.Grayscale()
    GetChannel(1)
])

# Create Datasets
testset = torchvision.datasets.CIFAR10(root='CIFAR-DATA', train = False, download=True, transform=transform)

# Load Datasets
test_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=True)

  0%|          | 0/170498071 [00:00<?, ?it/s]

Extracting CIFAR-DATA/cifar-10-python.tar.gz to CIFAR-DATA


In [ ]:
class NetCIFAR10(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 36, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(36, 36, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # output: 36, 16, 16 
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(36, 36, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(36, 36, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # output: 36, 8, 8
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(36, 36, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(36, 36, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # output: 36, 4, 4
        )

        self.fc1 = nn.Sequential(
            nn.Flatten(), 
            nn.Linear(36 * 4 * 4, 10)
        )
        
    def forward(self, xb):
        xb = self.conv1(xb)
        #print(xb.shape)
        xb = self.conv2(xb)
        #print(xb.shape)
        xb = self.conv3(xb)
        #print(xb.shape)
        xb = self.fc1(xb)
        
        return xb

In [ ]:
# Computes the average accuracy
def evaluate_fxp(model, val_loader):

    batch_accuracies = []   # Accuracies of each forwarded batch
    
    for images, labels in val_loader:
        outputs = model(images)                    # Generates batch predictions
        
        # Computes the batch accuracy
        _, preds = torch.max(torch.from_numpy(outputs), dim=1)
        accuracy = torch.tensor(torch.sum(preds == labels).item() / len(preds))
        
        # Append the batch accuracy
        batch_accuracies.append(accuracy)
    
    # Computes the epoch average accuracy
    accuracy = torch.stack(batch_accuracies).mean() # Average accuracy
    
    return accuracy

In [ ]:
# Create the fixed point model
def model_fxp(model, shift):
       
    layers = [
        # conv1
        fxp.Conv(model.conv1[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv1[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv2
        fxp.Conv(model.conv2[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv2[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv3
        fxp.Conv(model.conv3[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv3[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),
        
        # fc1
        fxp.Linear(model.fc1[1], shift, shift)       
    ]

    return fxp.ModelFxp(layers)

In [ ]:
# Load the quantized floating point model
model = NetCIFAR10()
model.load_state_dict(torch.load('train2_qat_sh5_84_48.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(5,10)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth() 
    print('\n')

#     Parameters          shift   accuracy
#  train2_qat_sh6_85_49.pt  6      84.27
#  train2_qat_sh6_85_49.pt  7      85.017

# train2_qat_sh5_84_61.pt   5      80.26
# train2_qat_sh5_84_61.pt   6      83.40
# train2_qat_sh5_84_61.pt   7      84.23

# train2_qat_sh5_84_48.pt   5      79.89  
# train2_qat_sh5_84_48.pt   6      83.28
# train2_qat_sh5_84_48.pt   7      84.12


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 5, accuracy: 0.7989517450332642
ConvLayers[0]: min = -193, max = 190 MIN_WIDTH = 9.0
ConvLayers[1]: min = -496, max = 333 MIN_WIDTH = 10.0
ConvLayers[2]: min = -677, max = 418 MIN_WIDTH = 11.0
ConvLayers[3]: min = -570, max = 523 MIN_WIDTH = 11.0
ConvLayers[4]: min = -564, max = 489 MIN_WIDTH = 11.0
ConvLayers[5]: min = -589, max = 502 MIN_WIDTH = 11.0
FullLayers[0]: min = -445, max = 1121 MIN_WIDTH = 12.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 2.0
ConvLayers[2] min width: 2.0
ConvLayers[3] min width: 2.0
ConvLayers[4] min width: 2.0
ConvLayers[5] min width: 2.0
FullLayers[0] min width: 2.0


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Con

In [ ]:
model = NetCIFAR10()
model.load_state_dict(torch.load('train2_qat_sh6_84_49.pt', map_location=torch.device('cpu')))

shift = 7   
model_test = model_fxp(model, shift)     
print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}\n")


In [ ]:
model_test.AccumulatorsWidth()
model_test.WeightsWidth()

In [ ]:
model = NetCIFAR10()
model.load_state_dict(torch.load('train2_qat_sh6_84_49.pt', map_location=torch.device('cpu')))

shift = 7   
model_test = model_fxp(model, shift)

img, _ = next(iter(test_loader))

In [ ]:
test = img[2]
test = torch.unsqueeze(test, 0)
test.shape

In [ ]:
#test = torch.full((1,1,32,32), 1)
#test = torch.full((1,1,10,10), 2).to(torch.float32)
#test = torch.randint(0, 1, (1,1,10,10)).to(torch.float32)
#test = torch.rand(1,1,32,32)
#test = torch.zeros(1,1,32,32)
vhex = np.vectorize(hex)
np.set_printoptions(threshold=np.inf)


out = model_test(test)

print(vhex(out.astype('uint32')))

file = open("test.txt", "w+")
bias_shift = shift
a = (test * (2 ** bias_shift)).numpy()
a = a.astype('int16')

for p in a.flatten():
    file.write('{0:0{1}X}\n'.format(p & 0xFFF, 3))
file.close()